In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings("ignore")

## 1. Load and Explore Data

In [ ]:
df = pd.read_csv(r"../data/credit_risk_dataset.csv")
print("Shape:", df.shape)
print("\nNull counts:")
print(df.isnull().sum())
df.head()

In [ ]:
print("Class distribution:")
print(df["loan_status"].value_counts())
print(df["loan_status"].value_counts(normalize=True).round(3))

## 2. Preprocessing

In [ ]:
# Ordinal encoding for loan_grade: A=1 (lowest risk/best grade) to G=7 (highest risk/worst grade)
grade_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
df["loan_grade"] = df["loan_grade"].map(grade_map)

y = df["loan_status"]
X = df.drop(columns=["loan_status"])

# One-hot encode remaining categoricals
X = pd.get_dummies(X)

# Impute missing values with column median
for col in X.columns:
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].median())

print("Feature matrix shape:", X.shape)
print("Features:", X.columns.tolist())

In [ ]:
# Stratified split to preserve class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, train_size=0.8, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)
print("Train class ratio:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Test class ratio: ", y_test.value_counts(normalize=True).round(3).to_dict())

## 3. Baseline Model

In [ ]:
baseline = RandomForestClassifier(random_state=42)
baseline.fit(X_train, y_train)

y_pred_baseline = baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred_baseline)
baseline_auc = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])

print(f"Baseline RandomForest — Accuracy: {baseline_acc:.4f} | ROC-AUC: {baseline_auc:.4f}")

## 4. Hyperparameter Tuning — Random Forest

In [ ]:
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": ["balanced", None],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    n_iter=20,
    cv=5,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
rf_search.fit(X_train, y_train)

print("Best RF params:", rf_search.best_params_)
print(f"Best CV ROC-AUC: {rf_search.best_score_:.4f}")

In [ ]:
best_rf = rf_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1])

print(f"Tuned RandomForest — Accuracy: {rf_acc:.4f} | ROC-AUC: {rf_auc:.4f}")

## 5. Hyperparameter Tuning — Gradient Boosting

In [ ]:
gb_param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5, 6],
    "min_samples_split": [2, 5, 10],
    "subsample": [0.7, 0.8, 0.9, 1.0],
}

gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_param_grid,
    n_iter=20,
    cv=5,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
gb_search.fit(X_train, y_train)

print("Best GB params:", gb_search.best_params_)
print(f"Best CV ROC-AUC: {gb_search.best_score_:.4f}")

In [ ]:
best_gb = gb_search.best_estimator_
y_pred_gb = best_gb.predict(X_test)
gb_acc = accuracy_score(y_test, y_pred_gb)
gb_auc = roc_auc_score(y_test, best_gb.predict_proba(X_test)[:, 1])

print(f"Tuned GradientBoosting — Accuracy: {gb_acc:.4f} | ROC-AUC: {gb_auc:.4f}")

## 6. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Baseline RandomForest", "Tuned RandomForest", "Tuned GradientBoosting"],
    "Accuracy": [baseline_acc, rf_acc, gb_acc],
    "ROC-AUC":  [baseline_auc, rf_auc, gb_auc],
})
results = results.sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

## 7. Best Model — Detailed Evaluation

In [ ]:
# Pick the model with the highest ROC-AUC
best_model_name = results.iloc[0]["Model"]
best_model = best_gb if "Gradient" in best_model_name else best_rf
y_pred_best = best_model.predict(X_test)

print(f"Best model: {best_model_name}\n")
print(classification_report(y_test, y_pred_best, target_names=["No Default (0)", "Default (1)"]))

In [ ]:
# Cross-validation on best model
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring="roc_auc", n_jobs=-1)
print(f"5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=["No Default", "Default"],
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title(f"Confusion Matrix\n{best_model_name}")

# Feature importance
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
    top10 = importances.nlargest(10)
    top10.sort_values().plot(kind="barh", ax=axes[1], color="steelblue")
    axes[1].set_title("Top 10 Feature Importances")
    axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()